In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import warnings


In [ ]:
df = pd.read_csv("real_estate_prices.csv")

print("Rows and columns:", df.shape)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.columns = [
    re.sub(r'[^a-zA-Z0-9]+', '_', col.strip()).strip('_').lower()
    for col in df.columns
]

if "unnamed_0" in df.columns:
    df.drop(columns=["unnamed_0"], inplace=True)

print("Cleaned column names:")
print(df.columns.tolist())

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

In [ ]:
df["bedrooms"] = df["bedrooms"].fillna(df["bedrooms"].median())
df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms"].median())

print("Missing values after handling:")
print(df.isnull().sum().sum())

In [ ]:
print("Duplicate rows before removal:", df.duplicated().sum())

df = df.drop_duplicates().copy()

print("Duplicate rows after removal:", df.duplicated().sum())

Data merging 

In [ ]:
neighborhood_data = pd.DataFrame({
    "zipcode": [98001, 98002, 98003],
    "avg_income": [65000, 58000, 72000]
})

merged_df = df.merge(
    neighborhood_data,
    on="zipcode",
    how="left"
)

print("Merged dataset:")
display(merged_df.head())

print("Merged dataset shape:", merged_df.shape)

Filter and subset data : Filter properties sold in 2014

In [ ]:
df["date"] = pd.to_datetime(
    df["date"],
    format="%Y%m%dT%H%M%S",
    errors="coerce"
)

df_2014 = df[df["date"].dt.year == 2014].copy()

print("Properties sold in 2014:", len(df_2014))
display(df_2014.head())

Filter by property price

In [ ]:
affordable_properties = df[df["price"] < 500000].copy()

print("Properties priced below $500,000:",
      len(affordable_properties))

display(affordable_properties.head())

Filter properties by location

In [ ]:
location_name = "Electronic City Phase II"

location_properties = df[
    df["location"] == location_name
]

print(location_properties.head())
print("Number of properties:", len(location_properties))

Encode categorical variables

In [ ]:
df_encoded = pd.get_dummies(
    df,
    columns=["waterfront"],
    dtype=int
)

print("Original dataset shape:", df.shape)
print("Encoded dataset shape:", df_encoded.shape)

display(df_encoded.head())

Aggregate data and calculate summary statistics

In [ ]:
price_by_zipcode = df.groupby("zipcode")["price"].agg(
    ["count", "mean", "min", "max"]
).round(2)

price_by_zipcode = price_by_zipcode.sort_values(
    by="mean",
    ascending=False
)

display(price_by_zipcode.head(10))

In [ ]:
price_by_bedrooms = df.groupby("bedrooms")["price"].agg(
    ["count", "mean"]
).round(2)

display(price_by_bedrooms)

Identify and handle outliers

In [ ]:
plt.figure(figsize=(8, 4))
plt.boxplot(df["price"])
plt.title("Boxplot of House Prices")
plt.ylabel("Price ($)")
plt.show()

In [ ]:
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

In [49]:
outliers = df[
    (df["price"] < lower_limit) |
    (df["price"] > upper_limit)
]

print("Number of price outliers:", len(outliers))

display(outliers[["price", "bedrooms", "bathrooms",
                  "sqft_living", "zipcode"]].head(10))

Number of price outliers: 1146


,price,bedrooms,bathrooms,sqft_living,zipcode
5,1225000.0,4.0,4.50,5420,98053
21,2000000.0,3.0,2.75,3050,98040
49,1350000.0,3.0,2.50,2753,98070
69,1325000.0,5.0,2.25,3200,98004
125,1450000.0,4.0,2.75,2750,98004
153,2250000.0,4.0,3.25,5180,98006
216,1505000.0,5.0,3.00,3300,98040
246,2400000.0,4.0,2.50,3650,98074
269,2900000.0,4.0,3.25,5050,98004
270,1365000.0,4.0,4.75,5310,98077


In [50]:
df_cleaned = df[
    (df["price"] >= lower_limit) &
    (df["price"] <= upper_limit)
].copy()

print("Original dataset shape:", df.shape)
print("Dataset shape after outlier removal:", df_cleaned.shape)

Original dataset shape: (21613, 21)
Dataset shape after outlier removal: (20467, 21)


In [51]:
print("Final dataset shape:", df_cleaned.shape)

print("\nMissing values:")
print(df_cleaned.isnull().sum().sum())

print("\nDuplicate rows:")
print(df_cleaned.duplicated().sum())

print("\nFinal dataset:")
display(df_cleaned.head())

print("\nStatistical summary:")
display(df_cleaned.describe())

Final dataset shape: (20467, 21)

Missing values:
0

Duplicate rows:
0

Final dataset:


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,2014-10-13,221900.0,3.0,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,2014-12-09,538000.0,3.0,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,2015-02-25,180000.0,2.0,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,2014-12-09,604000.0,4.0,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,2015-02-18,510000.0,3.0,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503



Statistical summary:


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
count,2.046700e+04,20467,2.046700e+04,20467.000000,20467.000000,20467.000000,2.046700e+04,20467.000000,20467.000000,20467.000000,...,20467.000000,20467.000000,20467.000000,20467.000000,20467.000000,20467.000000,20467.000000,20467.000000,20467.000000,20467.000000
mean,4.603223e+09,2014-10-29 04:55:51.111545344,4.769846e+05,3.331509,2.052621,1975.558167,1.461041e+04,1.476035,0.002980,0.172668,...,7.530561,1708.330972,267.227195,1970.811013,74.681145,98078.555773,47.556888,-122.213265,1922.255191,12447.084526
min,1.000102e+06,2014-05-02 00:00:00,7.500000e+04,1.000000,0.500000,290.000000,5.200000e+02,1.000000,0.000000,0.000000,...,1.000000,290.000000,0.000000,1900.000000,0.000000,98001.000000,47.155900,-122.519000,399.000000,651.000000
25%,2.131701e+09,2014-07-22 00:00:00,3.150000e+05,3.000000,1.500000,1400.000000,5.000000e+03,1.000000,0.000000,0.000000,...,7.000000,1170.000000,0.000000,1951.000000,0.000000,98033.000000,47.460400,-122.330000,1470.000000,5046.000000
50%,3.905082e+09,2014-10-16 00:00:00,4.375000e+05,3.000000,2.000000,1860.000000,7.500000e+03,1.000000,0.000000,0.000000,...,7.000000,1520.000000,0.000000,1974.000000,0.000000,98065.000000,47.567000,-122.232000,1800.000000,7542.000000
75%,7.338101e+09,2015-02-17 00:00:00,6.000000e+05,4.000000,2.500000,2431.000000,1.031900e+04,2.000000,0.000000,0.000000,...,8.000000,2100.000000,510.000000,1996.000000,0.000000,98118.000000,47.679800,-122.123000,2280.000000,9883.500000
max,9.900000e+09,2015-05-24 00:00:00,1.127500e+06,33.000000,7.500000,7480.000000,1.651359e+06,3.500000,1.000000,4.000000,...,12.000000,5710.000000,2720.000000,2015.000000,2015.000000,98199.000000,47.777600,-121.315000,5380.000000,871200.000000
std,2.877364e+09,NaN,2.083713e+05,0.909350,0.710616,774.833460,4.010956e+04,0.537033,0.054513,0.640950,...,1.039192,727.296461,409.031661,29.160540,378.761647,53.336953,0.141038,0.142412,614.932010,26538.592250


In [52]:
df_cleaned.to_csv(
    "cleaned_real_estate_prices.csv",
    index=False
)

print("Cleaned dataset saved successfully")

Cleaned dataset saved successfully
